# Trial32 event catalogs

Build earthquake-style snap, subavalanche, and avalanche catalogs directly from the modular `crumpling` pipeline. Snap magnitude is zero (unit event); cascade and avalanche magnitude is `log10(n_swaps)`.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "pyproject.toml").is_file() and (base / "crumpling").is_dir():
            return base
    raise FileNotFoundError(f"Could not locate the repository above {here}")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

import crumpling.catalogs
import crumpling.config
import crumpling.pipeline

DATA = ROOT / "data/raw/trial32/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt"
CONFIG = ROOT / "configs/trial32/swapstat.params.ini"
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
settings = crumpling.config.load_config(CONFIG)
result = crumpling.pipeline.analyze_file(DATA, settings)

snaps = crumpling.catalogs.snaps(result)
subav = crumpling.catalogs.subavalanches(result)
av = crumpling.catalogs.avalanches(result)

summary = pd.DataFrame(
    {
        "catalog": ["snaps", "subavalanches", "avalanches"],
        "events": [len(snaps), len(subav), len(av)],
        "start_time": [frame["time"].min() for frame in (snaps, subav, av)],
        "end_time": [frame["time"].max() for frame in (snaps, subav, av)],
        "median_dt": [frame["dt"].median() for frame in (snaps, subav, av)],
        "max_magnitude": [frame["magnitude"].max() for frame in (snaps, subav, av)],
    }
).set_index("catalog")
summary


In [ ]:
def plot_catalog(frame: pd.DataFrame, title: str) -> None:
    positive_dt = frame.loc[frame["dt"] > 0]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(title)
    axes[0, 0].scatter(frame["time"], frame["magnitude"], s=10, alpha=0.5)
    axes[0, 0].set(xlabel="time", ylabel="magnitude", title="Magnitude over time")
    axes[0, 1].plot(frame["time"], frame["cumulative_count"])
    axes[0, 1].set(xlabel="time", ylabel="events", title="Cumulative event number")
    spatial = axes[0, 2].scatter(frame["x"], frame["y"], c=frame["magnitude"], s=10)
    axes[0, 2].set(xlabel="x", ylabel="y", title="Spatial catalog")
    axes[0, 2].set_aspect("equal", adjustable="box")
    fig.colorbar(spatial, ax=axes[0, 2], label="magnitude")
    axes[1, 0].hist(frame["magnitude"], bins=30)
    axes[1, 0].set(title="Magnitude distribution")
    axes[1, 1].hist(positive_dt["log10_dt"], bins=35)
    axes[1, 1].set(xlabel="log10(dt)", title="Inter-event time distribution")
    axes[1, 2].scatter(positive_dt["magnitude"], positive_dt["log10_dt"], s=10)
    axes[1, 2].set(xlabel="magnitude", ylabel="log10(dt)", title="Magnitude vs dt")
    fig.tight_layout()


for frame, title in ((snaps, "Snaps"), (subav, "Subavalanches"), (av, "Avalanches")):
    plot_catalog(frame, title)
plt.show()
